<a href="https://colab.research.google.com/github/ngalvisarias/NLP-Domestic-Violence/blob/main/NLP_2025_Domestic_Violence_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p><img alt="Colaboratory logo" height="160px" src="https://www.staffnet.manchester.ac.uk/brand/visual-identity/logo/logo_big.gif" align="left" hspace="10px" vspace="0px"></p>

# **NLP Domestic Violence A**
**Author**: Natalia Galvis Arias  
**Data**: NUSE 2021, 2022, 2023
Day of the last modification : 04/07/2025

<p><a name="fp53"></a></p>

## **1. Libraries**

In [1]:
import numpy as np                        # for performing a wide variety of mathematical operations on arrays
import pandas as pd                       # for analyzing, cleaning, exploring, and manipulating data
import matplotlib.pyplot as plt           # for creating static, animated, and interactive visualizations
import seaborn as sns                     # for making statistical graphics in Python
#import nashpy as nash                     # for the computation of equilibria in game theory
import sklearn                            # for implementing machine learning models and statistical modelling

<p><a name="fp53"></a></p>

## **2. Import Data**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Define the folder path
folder_path = '/content/drive/MyDrive/CODE/NUSE'  # CODE - NUSE

# nuse_nlp DataFrame
nuse_nlp = pd.read_csv(folder_path + 'nuse_nlp.csv')

# nuse_sample_50 DataFrame
nuse_sample_50 = pd.read_csv(folder_path + 'nuse_sample_50.csv')

nuse_nlp.info()
nuse_nlp.head()

nuse_sample_50.info()
nuse_sample_50.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 376149 entries, 0 to 376148
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   No. Incidente      376149 non-null  object 
 1   Diferencia Tiempo  361832 non-null  object 
 2   Tipo Incidente     376149 non-null  int64  
 3   Latitud            376149 non-null  float64
 4   Longitud           376149 non-null  float64
 5   Localidad          369944 non-null  object 
 6   Comentarios        376105 non-null  object 
 7   Fecha              376149 non-null  object 
 8   Hora               376149 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 25.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   No. Incidente      50 non-null     object 
 1   Diferencia Tiempo  47 non-null     ob

,No. Incidente,Diferencia Tiempo,Tipo Incidente,Latitud,Longitud,Localidad,Comentarios,Fecha,Hora
0,SUR-01641668-23,00:00:12,934,4.620969,-74.212378,BOSA,LLAMANTE MANIFIESTA QUE EL AMIGO DE LA HIJA ES...,2023-09-14,09:21:28
1,SUR-00467859-22,00:00:15,611,4.695711,-74.061873,SUBA,INIDCA QUE CONTINUAMENTE LE PEGAN A UN M...,2022-03-17,21:01:30
2,SUR-01705640-22,00:00:07,934,4.719808,-74.130054,ENGATIVA,BARRIO EL UNIR-ESPOS AGRESIVO ROMPIO LOS VIDRI...,2022-09-14,23:19:20
3,SUR-00186387-22,00:00:58,934,0.000000,0.000000,USME,VAN HACIA EL CAI DE DANUBIO AZUL // FEMENINA E...,2022-02-02,00:28:12
4,SUR-02354627-22,00:00:16,934,4.747762,-74.048514,SUBA,INF SOBRE 934 ENTRE MADRE E HIJO EN AP,2022-12-18,09:51:27


<p><a name="fp53"></a></p>

## **3. Machine Learning**

In [9]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

In [10]:
import pandas as pd
import numpy as np
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Text processing and ML libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import joblib

# Download required NLTK data
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
except:
    pass

class SpanishTextPreprocessor(BaseEstimator, TransformerMixin):
    """
    Advanced Spanish text preprocessor for domestic violence risk assessment.
    Handles negation detection, regional variations, and domain-specific terminology.
    """

    def __init__(self):
        self.stemmer = SnowballStemmer('spanish')
        self.stop_words = set(stopwords.words('spanish'))

        # Enhanced keyword dictionaries
        self.high_risk_keywords = {
            'weapons': ['arma', 'disparo', 'revolver', 'tiros', 'pistola', 'puñal', 'bala',
                       'cuchillo', 'machete', 'navaja', 'escopeta', 'rifle', 'explosivo'],
            'violence': ['golpe', 'golpeando', 'golpear', 'agresion', 'agredir', 'atacar',
                        'matar', 'asesinar', 'violencia', 'brutal', 'salvaje', 'torturar'],
            'threats': ['amenaza', 'amenazar', 'intimidar', 'extorsion', 'chantaje'],
            'injuries': ['herida', 'herido', 'lesion', 'sangre', 'sangrado', 'ambulancia'  'fractura', 'moretones',
                        'contusion', 'traumatismo', 'hospitalizar'],
            'vulnerable': ['embarazada', 'gestante', 'niñas', 'niños', 'menor', 'anciano',
                          'discapacitado', 'bebe', 'infante'],
            'substances': ['embriaguez', 'alicorado', 'droga', 'narcótico', 'sustancia',
                          'alcohol', 'borracho', 'intoxicado'],
            'legal': ['medida', 'proteccion', 'denuncia', 'orden', 'restriccion', 'judicial'],
            'severity': ['urgente', 'grave', 'critico', 'emergencia', 'inmediato', 'extremo']
        }

        # Enhanced negation patterns
        self.negation_patterns = [
            r'\bno\s+(?:hay|tiene|porta|posee|presenta|registra|reporta)\b',
            r'\bsin\s+(?:armas|heridos|lesiones|violencia|agresion)\b',
            r'\bno\s+(?:se\s+)?(?:observa|evidencia|registra|reporta|sabe)\b',
            r'\bausencia\s+de\b',
            r'\bnegativa\s+(?:de|para)\b',
            r'\bno\s+(?:info|informacion|datos)\b',
            r'\bni\s+(?:armas|heridos|lesiones)\b',
            r'\bn\s+(?:armas|heridos)\b',  # Abbreviated negation
            r'\bno\s+(?:hay|existe|se\s+encuentra)\s+(?:evidencia|prueba|rastro)\b'
        ]

        # Regional variations and synonyms
        self.regional_variations = {
            'guaro': 'alcohol',
            'pola': 'cerveza',
            'punal': 'cuchillo',
            'fierro': 'arma',
            'cucho': 'anciano',
            'pelado': 'adolescente',
            'marido': 'esposo',
            'conyugue': 'esposo',
            'pareja': 'esposo',
            'femenina': 'mujer',
            'man': 'hombre',
            'golpear': 'agredir',
            'pegar': 'golpear',
            'porta': 'llevar_arma'
        }

    def detect_negation_context(self, text: str) -> List[Tuple[str, bool]]:
        """
        Detect negation context for each sentence.
        Returns list of (sentence, is_negated) tuples.
        """
        sentences = sent_tokenize(text, language='spanish')
        negated_sentences = []

        for sentence in sentences:
            is_negated = False
            sentence_lower = sentence.lower()

            # Check for negation patterns
            for pattern in self.negation_patterns:
                if re.search(pattern, sentence_lower):
                    is_negated = True
                    break

            negated_sentences.append((sentence, is_negated))

        return negated_sentences

    def normalize_regional_terms(self, text: str) -> str:
        """Replace regional terms with standard equivalents."""
        text_lower = text.lower()
        for regional, standard in self.regional_variations.items():
            text_lower = re.sub(r'\b' + regional + r'\b', standard, text_lower)
        return text_lower

    def extract_risk_features(self, text: str) -> Dict[str, int]:
        """
        Extract risk-related features from text, considering negation.
        """
        features = {}

        # Initialize feature counts
        for category in self.high_risk_keywords:
            features[f'{category}_count'] = 0
            features[f'{category}_negated_count'] = 0

        # Process each sentence separately for negation context
        negated_sentences = self.detect_negation_context(text)

        for sentence, is_negated in negated_sentences:
            sentence_normalized = self.normalize_regional_terms(sentence)
            words = word_tokenize(sentence_normalized, language='spanish')

            # Count keywords in each category
            for category, keywords in self.high_risk_keywords.items():
                for keyword in keywords:
                    if keyword in sentence_normalized:
                        if is_negated:
                            features[f'{category}_negated_count'] += 1
                        else:
                            features[f'{category}_count'] += 1

        return features

    def clean_text(self, text: str) -> str:
        """Basic text cleaning and normalization."""
        if pd.isna(text):
            return ""

        # Convert to lowercase
        text = text.lower()

        # Remove special characters but keep Spanish characters
        text = re.sub(r'[^\w\sáéíóúñü]', ' ', text)

        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        # Normalize regional terms
        text = self.normalize_regional_terms(text)

        return text

    def preprocess_text(self, text: str) -> str:
        """
        Complete text preprocessing pipeline.
        """
        # Clean text
        text = self.clean_text(text)

        # Tokenize
        words = word_tokenize(text, language='spanish')

        # Remove stopwords and stem
        processed_words = []
        for word in words:
            if word not in self.stop_words and len(word) > 2:
                # Use stemming for better generalization
                stemmed = self.stemmer.stem(word)
                processed_words.append(stemmed)

        return ' '.join(processed_words)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.Series):
            return X.apply(self.preprocess_text)
        else:
            return [self.preprocess_text(text) for text in X]

class DomesticViolenceRiskClassifier:
    """
    Complete ML pipeline for domestic violence risk classification.
    """

    def __init__(self):
        self.preprocessor = SpanishTextPreprocessor()
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.8,
            sublinear_tf=True
        )
        self.label_encoder = LabelEncoder()
        self.classifier = None
        self.pipeline = None

    def create_synthetic_labels(self, df: pd.DataFrame) -> pd.Series:
        """
        Create synthetic risk labels based on keyword analysis.
        This is a placeholder - replace with actual labeled data when available.
        """
        labels = []

        for _, row in df.iterrows():
            text = str(row['Comentarios']) if pd.notna(row['Comentarios']) else ""

            # Extract features
            features = self.preprocessor.extract_risk_features(text)

            # Calculate risk score
            high_risk_score = (
                features['weapons_count'] * 3 +
                features['violence_count'] * 2 +
                features['threats_count'] * 2 +
                features['injuries_count'] * 2 +
                features['vulnerable_count'] * 2 +
                features['severity_count'] * 1
            )

            # Subtract negated risks
            negated_penalty = sum(v for k, v in features.items() if 'negated_count' in k)
            final_score = max(0, high_risk_score - negated_penalty)

            # Assign risk category
            if final_score >= 5:
                labels.append('High Risk')
            elif final_score >= 2:
                labels.append('Moderate Risk')
            else:
                labels.append('Low Risk')

        return pd.Series(labels)

    def train_classifier(self, X_train, y_train, classifier_type='random_forest'):
        """
        Train the risk classification model.
        """
        # Create pipeline
        if classifier_type == 'random_forest':
            classifier = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                class_weight='balanced'
            )
        elif classifier_type == 'logistic_regression':
            classifier = LogisticRegression(
                random_state=42,
                max_iter=1000,
                class_weight='balanced'
            )
        elif classifier_type == 'svm':
            classifier = SVC(
                kernel='rbf',
                random_state=42,
                class_weight='balanced',
                probability=True
            )
        else:
            raise ValueError(f"Unknown classifier type: {classifier_type}")

        # Create pipeline
        self.pipeline = Pipeline([
            ('preprocessor', self.preprocessor),
            ('vectorizer', self.vectorizer),
            ('classifier', classifier)
        ])

        # Fit the pipeline
        self.pipeline.fit(X_train, y_train)

        return self.pipeline

    def evaluate_model(self, X_test, y_test):
        """
        Evaluate the trained model.
        """
        if self.pipeline is None:
            raise ValueError("Model not trained. Call train_classifier first.")

        # Make predictions
        y_pred = self.pipeline.predict(X_test)
        y_pred_proba = self.pipeline.predict_proba(X_test)

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)

        print(f"Model Accuracy: {accuracy:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))

        return {
            'accuracy': accuracy,
            'predictions': y_pred,
            'probabilities': y_pred_proba,
            'classification_report': classification_report(y_test, y_pred, output_dict=True)
        }

    def predict_risk(self, texts):
        """
        Predict risk for new texts.
        """
        if self.pipeline is None:
            raise ValueError("Model not trained. Call train_classifier first.")

        predictions = self.pipeline.predict(texts)
        probabilities = self.pipeline.predict_proba(texts)

        return predictions, probabilities

    def save_model(self, filepath):
        """Save the trained model."""
        if self.pipeline is None:
            raise ValueError("Model not trained. Call train_classifier first.")

        joblib.dump(self.pipeline, filepath)

    def load_model(self, filepath):
        """Load a trained model."""
        self.pipeline = joblib.load(filepath)

# Example usage and training function
def train_and_evaluate_model(df: pd.DataFrame, test_size: float = 0.2):
    """
    Complete training and evaluation pipeline.
    """
    print("🚀 Starting Domestic Violence Risk Classification Pipeline")
    print(f"📊 Dataset shape: {df.shape}")

    # Initialize classifier
    classifier = DomesticViolenceRiskClassifier()

    # Create synthetic labels (replace with actual labels when available)
    print("🏷️ Creating synthetic risk labels...")
    y = classifier.create_synthetic_labels(df)

    print(f"📈 Risk distribution:")
    print(y.value_counts().sort_index())

    # Prepare features
    X = df['Comentarios'].fillna('')

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    print(f"🔄 Training set size: {len(X_train)}")
    print(f"🔄 Test set size: {len(X_test)}")

    # Train different classifiers
    classifiers = ['random_forest', 'logistic_regression']
    best_accuracy = 0
    best_model = None

    for clf_name in classifiers:
        print(f"\n🤖 Training {clf_name}...")

        # Train classifier
        classifier.train_classifier(X_train, y_train, clf_name)

        # Evaluate
        results = classifier.evaluate_model(X_test, y_test)

        if results['accuracy'] > best_accuracy:
            best_accuracy = results['accuracy']
            best_model = classifier.pipeline

    # Set the best model
    classifier.pipeline = best_model

    print(f"\n✅ Best model accuracy: {best_accuracy:.4f}")

    return classifier, X_test, y_test

# Function to analyze individual cases
def analyze_case(classifier, text: str):
    """
    Analyze a single case and provide detailed risk assessment.
    """
    # Get prediction
    prediction, probabilities = classifier.predict_risk([text])

    # Get feature analysis
    features = classifier.preprocessor.extract_risk_features(text)

    # Get negation analysis
    negated_sentences = classifier.preprocessor.detect_negation_context(text)

    print(f"📄 Text: {text[:100]}...")
    print(f"🎯 Predicted Risk: {prediction[0]}")
    print(f"📊 Probabilities: {dict(zip(classifier.pipeline.classes_, probabilities[0]))}")
    print(f"🔍 Risk Features Found:")

    for category, count in features.items():
        if count > 0:
            print(f"  - {category}: {count}")

    print(f"🚫 Negation Analysis:")
    for sentence, is_negated in negated_sentences:
        if is_negated:
            print(f"  - NEGATED: {sentence}")

# Main execution with your nuse_nlp dataset
if __name__ == "__main__":
    # Use your nuse_nlp DataFrame directly
    print("📊 Working with nuse_nlp dataset...")
    print(f"Dataset shape: {nuse_nlp.shape}")
    print(f"Columns: {list(nuse_nlp.columns)}")

    # Check for missing values in Comentarios
    missing_comments = nuse_nlp['Comentarios'].isna().sum()
    print(f"Missing comments: {missing_comments} ({missing_comments/len(nuse_nlp)*100:.2f}%)")

    # Train and evaluate the model
    classifier, X_test, y_test = train_and_evaluate_model(nuse_nlp)

    # Save the trained model
    classifier.save_model('domestic_violence_risk_model.pkl')
    print("💾 Model saved as 'domestic_violence_risk_model.pkl'")

    # Analyze some sample cases from your dataset
    print("\n🔍 Analyzing sample cases from your dataset:")
    sample_indices = nuse_nlp.sample(3).index

    for idx in sample_indices:
        case = nuse_nlp.loc[idx]
        text = str(case['Comentarios']) if pd.notna(case['Comentarios']) else ""

        print(f"\n📋 Case {case['No. Incidente']}:")
        print(f"📅 Date: {case['Fecha']} {case['Hora']}")
        print(f"📍 Location: {case['Localidad']}")

        if text:
            analyze_case(classifier, text)
        else:
            print("❌ No comment text available")

        print("-" * 60)

    # Generate predictions for the entire dataset
    print("\n🎯 Generating predictions for entire dataset...")
    all_predictions, all_probabilities = classifier.predict_risk(nuse_nlp['Comentarios'].fillna(''))

    # Add predictions to the dataset
    nuse_nlp_with_predictions = nuse_nlp.copy()
    nuse_nlp_with_predictions['Risk_Category'] = all_predictions
    nuse_nlp_with_predictions['High_Risk_Probability'] = all_probabilities[:, 0]  # Adjust index based on class order
    nuse_nlp_with_predictions['Moderate_Risk_Probability'] = all_probabilities[:, 1]
    nuse_nlp_with_predictions['Low_Risk_Probability'] = all_probabilities[:, 2]

    # Show risk distribution
    print("\n📈 Risk Distribution Across Dataset:")
    risk_counts = pd.Series(all_predictions).value_counts()
    for risk_level, count in risk_counts.items():
        percentage = (count / len(all_predictions)) * 100
        print(f"  {risk_level}: {count:,} cases ({percentage:.1f}%)")

    # Show high-risk cases by location
    print("\n🗺️ High-Risk Cases by Location:")
    high_risk_cases = nuse_nlp_with_predictions[nuse_nlp_with_predictions['Risk_Category'] == 'High Risk']
    if len(high_risk_cases) > 0:
        location_risk = high_risk_cases['Localidad'].value_counts().head(10)
        for location, count in location_risk.items():
            print(f"  {location}: {count:,} high-risk cases")

    # Show temporal patterns
    print("\n📅 High-Risk Cases by Hour:")
    if len(high_risk_cases) > 0:
        # Extract hour from time string
        high_risk_cases['Hour'] = pd.to_datetime(high_risk_cases['Hora'], format='%H:%M:%S').dt.hour
        hour_risk = high_risk_cases['Hour'].value_counts().sort_index()
        for hour, count in hour_risk.head(10).items():
            print(f"  {hour:02d}:00: {count:,} cases")

    print(f"\n✅ Analysis complete! Dataset with predictions saved to 'nuse_nlp_with_predictions'")
    print(f"🎯 High-risk cases identified: {len(high_risk_cases):,}")

# Save results to CSV
nuse_nlp_with_predictions.to_csv('nuse_nlp_with_risk_predictions.csv', index=False)
print("💾 Results saved to 'nuse_nlp_with_risk_predictions.csv'")

📊 Working with nuse_nlp dataset...
Dataset shape: (376149, 9)
Columns: ['No. Incidente', 'Diferencia Tiempo', 'Tipo Incidente', 'Latitud', 'Longitud', 'Localidad', 'Comentarios', 'Fecha', 'Hora']
Missing comments: 44 (0.01%)
🚀 Starting Domestic Violence Risk Classification Pipeline
📊 Dataset shape: (376149, 9)
🏷️ Creating synthetic risk labels...
📈 Risk distribution:
High Risk        133201
Low Risk         163373
Moderate Risk     79575
Name: count, dtype: int64
🔄 Training set size: 300919
🔄 Test set size: 75230

🤖 Training random_forest...
Model Accuracy: 0.6550

Classification Report:
               precision    recall  f1-score   support

    High Risk       0.80      0.82      0.81     26640
     Low Risk       0.94      0.37      0.54     32675
Moderate Risk       0.44      0.95      0.60     15915

     accuracy                           0.65     75230
    macro avg       0.72      0.72      0.65     75230
 weighted avg       0.78      0.65      0.65     75230


Confusion Matrix